# South Korea Trade in 5 Minutes - HS6 Quickstart

A compact starter analysis for **South Korea Customs Trade 2012-2026 - HSK10**.

This notebook intentionally starts from the analyst-friendly **HS6 monthly table** rather than loading all 22M+ HSK10 rows. It demonstrates the dataset structure, recent trade trends, partner concentration, product groups, and the audit files that preserve source anomalies instead of silently cleaning them away.

**Data grain used here:** `month x partner country x HS6`  
**Source:** Korea Customs Service public data; see the dataset's `SOURCES.md` and `METHODOLOGY.md` for provenance and caveats.


In [ ]:
from pathlib import Path
import pandas as pd
import pyarrow.dataset as ds
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA = None
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    # Kaggle may mount a private Dataset under an internal folder name that is
    # not identical to its public slug. Discover it by a required release file.
    matches = list(kaggle_input.rglob("trade_hs6_monthly.parquet"))
    for match in matches:
        candidate = match.parent
        if (candidate / "country_reference.csv").exists():
            DATA = candidate
            break

if DATA is None:
    local_candidates = [
        Path.cwd() / "release" / "kaggle",
        Path.cwd().parent.parent / "release" / "kaggle",
    ]
    DATA = next((path for path in local_candidates if path.exists()), None)

if DATA is None:
    mounted = [str(path) for path in kaggle_input.iterdir()] if kaggle_input.exists() else []
    raise FileNotFoundError(f"Could not locate the Kaggle dataset files. Mounted inputs: {mounted}")

print("Data path:", DATA)
print("Dataset files:", len(list(DATA.iterdir())))


## 1. Load only what we need

The HS6 file contains the full history, but this quickstart reads only **2024 onward** and only the columns used below. PyArrow can push the month filter into the Parquet scan instead of materializing the entire file.


In [ ]:
hs6_path = DATA / "trade_hs6_monthly.parquet"
hs6 = ds.dataset(hs6_path, format="parquet")

recent = hs6.to_table(
    filter=ds.field("month") >= "202401",
    columns=[
        "month", "country_code", "hs6", "hs4", "hs2",
        "export_usd", "import_usd", "trade_balance_usd",
        "has_residual", "exception_row_count",
    ],
).to_pandas()

countries = pd.read_csv(
    DATA / "country_reference.csv",
    dtype={"country_code": "string", "un_m49": "string", "iso_alpha3": "string"},
)

# The KCS codebook contains historical/special codes and an EU aggregate code.
# For partner rankings/trends, use current UN geographic matches plus Taiwan (TW)
# so aggregate/special codes do not create accidental double counting.
current_geo_codes = set(
    countries.loc[countries["un_match_status"].eq("matched_current_un"), "country_code"]
)
current_geo_codes.add("TW")
recent_geo = recent[recent["country_code"].isin(current_geo_codes)].copy()

print(f"Recent HS6 rows: {len(recent):,}")
print(f"Geographic-partner rows used for charts: {len(recent_geo):,}")
print(f"Months: {recent['month'].min()} -> {recent['month'].max()}")
print(f"Partner codes in full recent slice: {recent['country_code'].nunique():,}")
recent.head()


## 2. Recent trade with current geographic partners

The KCS reference includes current countries/territories **and** a small number of historical, special, or aggregate codes (for example `EU`, `ZZ`, and `Z1`). To avoid accidental double counting, the charts below use current UN geographic matches plus `TW` (Taiwan).

This is an **analytical partner universe**, not a claim that summing these rows exactly reproduces an independently published Korea-wide customs total. Values are reported in USD; exports use FOB basis and imports use CIF/customs value as documented in the dataset.


In [ ]:
monthly = (
    recent_geo.groupby("month", as_index=False)[["export_usd", "import_usd"]]
    .sum()
    .sort_values("month")
)
monthly["export_usd_bn"] = monthly["export_usd"] / 1e9
monthly["import_usd_bn"] = monthly["import_usd"] / 1e9

ax = monthly.plot(
    x="month", y=["export_usd_bn", "import_usd_bn"],
    figsize=(12, 5), marker="o"
)
ax.set_title("South Korea trade with current geographic partner codes")
ax.set_ylabel("USD billions")
ax.set_xlabel("Month")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

monthly.tail(8)


## 3. Top export partners in the latest available month

For plotting only, a partner without an English reference name is displayed by its KCS code (for example `TW`) rather than requiring a Korean font in the chart.


In [ ]:
latest_month = recent_geo["month"].max()
latest = recent_geo[recent_geo["month"] == latest_month]

partners = (
    latest.groupby("country_code", as_index=False)[["export_usd", "import_usd"]]
    .sum()
    .merge(
        countries[["country_code", "country_name_en", "country_name_ko"]],
        on="country_code", how="left"
    )
    .sort_values("export_usd", ascending=False)
)
partners["partner_name"] = partners["country_name_en"].fillna("").astype(str).str.strip()
missing_name = partners["partner_name"].eq("")
partners.loc[missing_name, "partner_name"] = partners.loc[missing_name, "country_code"]
partners["export_usd_bn"] = partners["export_usd"] / 1e9
partners["import_usd_bn"] = partners["import_usd"] / 1e9

print("Latest month:", latest_month)
partners[["country_code", "partner_name", "export_usd_bn", "import_usd_bn"]].head(15)


In [ ]:
top = partners.head(12).sort_values("export_usd_bn")
ax = top.plot.barh(
    x="partner_name", y="export_usd_bn", figsize=(9, 6), legend=False
)
ax.set_title(f"Top export partners - {latest_month}")
ax.set_xlabel("Exports (USD billions)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 4. Which product groups lead exports?

HS6 is internationally comparable, while `hs4` is useful for a more compact industry view. Below we aggregate the latest month to HS4 directly from the HS6 table.


In [ ]:
products_hs4 = (
    latest.groupby("hs4", as_index=False)[["export_usd", "import_usd"]]
    .sum()
    .sort_values("export_usd", ascending=False)
)
products_hs4["export_usd_bn"] = products_hs4["export_usd"] / 1e9
products_hs4["import_usd_bn"] = products_hs4["import_usd"] / 1e9
products_hs4.head(15)


## 5. Semiconductor example: HS4 `8542`

`8542` covers electronic integrated circuits. Because this notebook starts from HS6, we can still inspect both the HS6 subheadings and the destination mix without loading HSK10.


In [ ]:
semi = latest[latest["hs4"] == "8542"]

semi_hs6 = (
    semi.groupby("hs6", as_index=False)[["export_usd", "import_usd"]]
    .sum()
    .sort_values("export_usd", ascending=False)
)
semi_hs6["export_usd_bn"] = semi_hs6["export_usd"] / 1e9
semi_hs6.head(10)


In [ ]:
semi_partners = (
    semi.groupby("country_code", as_index=False)["export_usd"]
    .sum()
    .merge(
        countries[["country_code", "country_name_en", "country_name_ko"]],
        on="country_code", how="left"
    )
    .sort_values("export_usd", ascending=False)
)
semi_partners["partner_name"] = semi_partners["country_name_en"].fillna("").astype(str).str.strip()
missing_name = semi_partners["partner_name"].eq("")
semi_partners.loc[missing_name, "partner_name"] = semi_partners.loc[missing_name, "country_code"]
semi_partners["export_usd_bn"] = semi_partners["export_usd"] / 1e9
semi_partners[["country_code", "partner_name", "export_usd_bn"]].head(15)


## 6. Source fidelity: residuals and warning files

The release does **not** force every upstream row into a clean HSK10 shape. Rare shorter HS-code rows are kept in a separate exception file, and rare negative reported weights are preserved in a warning file.

Derived HS tables include safely mappable residuals only when the observed prefix identifies that level without guessing.


In [ ]:
exceptions = pd.read_parquet(DATA / "source_code_exceptions.parquet")
weight_warnings = pd.read_parquet(DATA / "source_weight_warnings.parquet")

print(f"Non-HSK10 source exceptions: {len(exceptions):,}")
print(f"Negative source-weight warnings: {len(weight_warnings):,}")
print(f"HS6 rows with mapped residuals in recent slice: {recent['has_residual'].sum():,}")

exceptions.head()


## Where to go next

- **HS2** - fast macro exploration
- **HS4** - industry analysis
- **HS6** - default choice for international product comparisons
- **HS8** - Korean national-detail prefix analysis
- **HSK10** - maximum Korean product/supply-chain detail

Useful next analyses:
1. export-market concentration by product,
2. semiconductor dependence by destination,
3. product diversification by partner,
4. revision-aware HSK10 supply-chain analysis,
5. forecasting monthly exports by HS6 and partner.

If you build something useful with this dataset, linking your public Notebook back to the dataset helps other analysts discover reproducible examples.
